# PCB Defect Detection — YOLOv8 train (Google Colab, bepul GPU)

Runtime → Change runtime type → **GPU** (T4) tanlang, so'ng kataklarni ketma-ket ishga tushiring.

In [ ]:
!pip install ultralytics roboflow -q

## 1. Dataset yuklab olish

`YOUR_API_KEY`, `workspace`, `project`, `version` qiymatlarini Roboflow Universe'dagi
tanlagan dataset sahifasidan ("show download code") ko'chiring.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("bare-pcb-defects").project("obj-detection-pcb-defects-yolov8")
version = project.version(1)
dataset = version.download("yolov8")
print(dataset.location)

## 2. Train qilish

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    project="runs",
    name="pcb_train",
)

## 3. Natijani baholash

In [ ]:
metrics = model.val()
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

## 4. best.pt'ni yuklab olish

Yuklangan faylni kompyuteringizda `backend/app/ml/best.pt` yo'liga joylashtiring,
so'ng `docker compose up -d --build backend` bilan qayta ishga tushiring.

In [ ]:
from google.colab import files
files.download("runs/pcb_train/weights/best.pt")